# FCV v2.2 — corrida robusta grado-publicacion (resumible en Drive)
Floor-Ceiling Validation: 8 metodos factoriales (floor, ILS, {GA,bPSO,bDE}x{vanilla,memetico} con LS compartida = descenso completo del mejor) sobre instancias DURAS reales (Pisinger knapsack n hasta 120, SCP, UFLP), 31 seeds, budget 1e4*dim, + control de decoder (penalty vs repair). Ver EXPERIMENT_DESIGN_v2.md.

**v2.2**: instancias knapsack con seed determinista (fix del bug hash() entre procesos) + operador memetico unificado + n=120 para el argumento de escala. SIG_VER nuevo => recomputa todo limpio.

**Resumible ante desconexiones**: el cache vive en Google Drive (`MyDrive/fcv_cache_v2/`). Si Colab se corta, re-ejecuta todas las celdas y continua.

In [ ]:
# 1) Repo
import os
if not os.path.isdir('fcv'):
    !git clone --depth 1 https://github.com/GastonGriott/fcv.git
%cd fcv
!git pull --ff-only
import sys; sys.path.insert(0, os.getcwd())
print('cpus:', os.cpu_count())

In [ ]:
# 2) Cache en Drive (sobrevive desconexiones)
from google.colab import drive
drive.mount('/content/drive')
import glob
DRIVE_CACHE = '/content/drive/MyDrive/fcv_cache_v2'
os.makedirs(DRIVE_CACHE, exist_ok=True)
os.environ['FCV_CACHE_V2'] = DRIVE_CACHE   # heredado por los workers (fork/spawn)
import fcv.run_v2 as r
print('CACHE_DIR =', r.CACHE_DIR, '| SIG', r.SIG_VER)
print('cache en Drive:', len(glob.glob(DRIVE_CACHE+'/*.json')), 'celdas')

In [ ]:
# 3) Corrida principal (8 metodos, knapsack/scp/uflp, 3 inst, 31 seeds). Reanuda desde cache.
specs = r.default_specs(instances=3)
print(len(specs), 'specs x 8 metodos x 31 seeds')
r.run(jobs=-1, specs=specs, n_seeds=31, out_prefix='/content/drive/MyDrive/fcv_v2')

In [ ]:
# 4) Sub-estudio decoder (penalty vs repair) — incluye n=120 para 'reaparece a escala'
ctl = [('knapsack', t, n, i) for t in ['strongly','mstr','spanner'] for n in [30,50,80,120] for i in range(3)]
ctl += [('scp','',n,i) for n in [60,90,120] for i in range(3)]
r.run(jobs=-1, specs=ctl, decoders=['penalty','repair'],
      methods_list=('floor','ils','mga'), n_seeds=31, out_prefix='/content/drive/MyDrive/fcv_v2_decoder')

In [ ]:
# 5) Descargar CSV (ya quedan en MyDrive/)
from google.colab import files
for f in ['fcv_v2_perseed.csv','fcv_v2_agg.csv','fcv_v2_decoder_perseed.csv','fcv_v2_decoder_agg.csv']:
    p = '/content/drive/MyDrive/' + f
    if os.path.exists(p): files.download(p)